In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("/content/Coursera.csv")

print("Before:", df.shape)
print(df.columns.tolist())

df = df.drop_duplicates()
df = df.dropna(how="all")

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("�", "'", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )


df = df.replace(["nan", "NaN", "None", "NULL", "", " "], np.nan)

df = df.dropna(subset=[
    "Course Name",
    "University",
    "Difficulty Level",
    "Course Rating",
    "Course URL",
    "Course Description",
    "Skills"
])


df["Course Rating"] = pd.to_numeric(df["Course Rating"], errors="coerce")
df = df.dropna(subset=["Course Rating"])

df["Difficulty Level"] = df["Difficulty Level"].str.title().str.strip()


valid_levels = [
    "Beginner",
    "Intermediate",
    "Advanced",
    "Mixed",
    "Not Calibrated"
]

df = df[df["Difficulty Level"].isin(valid_levels)]

df = df.drop_duplicates(subset=["Course Name", "University"], keep="first")


it_keywords = [

    "computer", "information technology", "software", "programming",
    "developer", "development", "coding", "code", "algorithm",
    "data structure", "web", "app", "application",

    "python", "java", "javascript", "typescript", "c++", "c#", "php",
    "html", "css", "react", "angular", "node", "django", "flutter",

    "data", "database", "sql", "mysql", "mongodb", "analytics",
    "machine learning", "deep learning", "artificial intelligence",
    "ai", "nlp", "computer vision", "pandas", "numpy",

    "cybersecurity", "security", "network", "cloud", "aws", "azure",
    "google cloud", "devops", "linux", "unix", "operating system",

    "computer-science", "information-technology", "data-science",
    "software-development", "data-management", "mobile-and-web-development"
]

pattern = "|".join([re.escape(k) for k in it_keywords])

search_text = (
    df["Course Name"].fillna("") + " " +
    df["Course Description"].fillna("") + " " +
    df["Skills"].fillna("")
).str.lower()

df_it = df[search_text.str.contains(pattern, regex=True)].copy()

df_it["Skills"] = (
    df_it["Skills"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_it["Course Description"] = (
    df_it["Course Description"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_it = df_it.sort_values(
    by=["Course Rating", "Course Name"],
    ascending=[False, True]
).reset_index(drop=True)

print("After IT cleaning:", df_it.shape)
display(df_it.head())

df_it.to_csv("/content/Coursera_IT_cleaned.csv", index=False)

print("Saved: /content/Coursera_IT_cleaned.csv")